In [ ]:
# IMPORTS
import imageio_ffmpeg
import cv2
from scipy.spatial.transform import Rotation as R

from egomimic.rldb.utils import S3RLDBDataset
import torch
import numpy as np
from scipy.spatial.transform import Rotation as R
from egomimic.utils.egomimicUtils import CameraTransforms, draw_actions, cam_frame_to_cam_pixels
import torchvision.io as io
import os
from egomimic.algo.hpt import HPT
from egomimic.utils.egomimicUtils import nds
import mediapy as mpy
import imageio_ffmpeg
mpy.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())

## Aria Data

In [ ]:
dataset = S3RLDBDataset(
    filters={"episode_hash": "2026-01-22-01-10-22-624000"}, mode="total", cache_root="/coc/flash7/skareer6/CacheEgoVerse/.cache", embodiment="aria_bimanual"
)

In [ ]:
dataset = S3RLDBDataset(
    filters={"episode_hash": "697c1e6c0cac8cd3c4873844"}, mode="total", cache_root="/coc/flash7/scratch/.cache", embodiment="scale_bimanual"
)

In [ ]:
# Make data_loader
data_loader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

In [ ]:
camera_transforms = CameraTransforms(intrinsics_key="scale", extrinsics_key="scale")

In [ ]:
def _split_action_pose(actions):
    # 14D layout: [L xyz ypr g, R xyz ypr g]
    # 12D layout: [L xyz ypr, R xyz ypr]
    if actions.shape[-1] == 14:
        left_xyz = actions[:, :3]
        left_ypr = actions[:, 3:6]
        right_xyz = actions[:, 7:10]
        right_ypr = actions[:, 10:13]
    elif actions.shape[-1] == 12:
        left_xyz = actions[:, :3]
        left_ypr = actions[:, 3:6]
        right_xyz = actions[:, 6:9]
        right_ypr = actions[:, 9:12]
    else:
        raise ValueError(f"Unsupported action dim {actions.shape[-1]}")
    return left_xyz, left_ypr, right_xyz, right_ypr


def viz_lerobot_xyz(im, actions, intrinsics):
    left_xyz, _, right_xyz, _ = _split_action_pose(actions)
    vis = draw_actions(
        im.copy(), "xyz", "Blues", left_xyz, None, intrinsics, arm="left"
    )
    vis = draw_actions(
        vis, "xyz", "Reds", right_xyz, None, intrinsics, arm="right"
    )
    return vis


def viz_lerobot_ypr(im, actions, intrinsics, axis_len_m=0.04):
    left_xyz, left_ypr, right_xyz, right_ypr = _split_action_pose(actions)
    vis = im.copy()

    def _draw_rotation_at_palm(frame, xyz_seq, ypr_seq, label, anchor_color):
        if len(xyz_seq) == 0 or len(ypr_seq) == 0:
            return frame

        palm_xyz = xyz_seq[0]
        palm_ypr = ypr_seq[0]
        rot = R.from_euler("ZYX", palm_ypr, degrees=False).as_matrix()

        axis_points_cam = np.vstack([
            palm_xyz,
            palm_xyz + rot[:, 0] * axis_len_m,
            palm_xyz + rot[:, 1] * axis_len_m,
            palm_xyz + rot[:, 2] * axis_len_m,
        ])

        px = cam_frame_to_cam_pixels(axis_points_cam, intrinsics)[:, :2]
        if not np.isfinite(px).all():
            return frame

        pts = np.round(px).astype(np.int32)
        h, w = frame.shape[:2]
        x0, y0 = pts[0]
        if not (0 <= x0 < w and 0 <= y0 < h):
            return frame

        cv2.circle(frame, (x0, y0), 4, anchor_color, -1)
        axis_colors = [(0, 0, 255), (0, 255, 0), (255, 0, 0)]  # x,y,z

        for i, color in enumerate(axis_colors, start=1):
            x1, y1 = pts[i]
            if 0 <= x1 < w and 0 <= y1 < h:
                cv2.line(frame, (x0, y0), (x1, y1), color, 2)
                cv2.circle(frame, (x1, y1), 2, color, -1)

        cv2.putText(
            frame, label, (x0 + 6, max(12, y0 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.4, anchor_color, 1, cv2.LINE_AA,
        )
        return frame

    vis = _draw_rotation_at_palm(vis, left_xyz, left_ypr, "L rot", (255, 180, 80))
    vis = _draw_rotation_at_palm(vis, right_xyz, right_ypr, "R rot", (80, 180, 255))
    return vis

In [ ]:
ims = []
image_key = "observations.images.front_img_1"
actions_key = "actions_ee_cartesian_cam"
for i, data in enumerate(data_loader):
    im = data[image_key][0].permute(1, 2, 0).cpu().numpy() * 255
    im = im.astype(np.uint8)
    actions = data[actions_key][0].cpu().numpy()
    im = viz_lerobot_xyz(im, actions, camera_transforms.intrinsics)
    ims.append(im)
    if i > 200:
        break

mpy.show_video(ims, fps=30)

In [ ]:

# Separate LeRobot YPR video (same logic, palm-centered projection)
ims_ypr = []
image_key = "observations.images.front_img_1"
actions_key = "actions_ee_cartesian_cam"
for i, data in enumerate(data_loader):
    im = data[image_key][0].permute(1, 2, 0).cpu().numpy() * 255
    im = im.astype(np.uint8)
    
    actions = data[actions_key][0].cpu().numpy()
    im_ypr = viz_lerobot_ypr(im, actions, camera_transforms.intrinsics)
    ims_ypr.append(im_ypr)
    if i > 200:
        break

mpy.show_video(ims_ypr, fps=30)

## Eva Data

In [ ]:
dataset = S3RLDBDataset(
    filters={"episode_hash": "2026-01-26-20-58-18-408000"}, mode="total", cache_root="/coc/flash7/skareer6/CacheEgoVerse/.cache", embodiment="eva_bimanual"
)

In [ ]:
# Make data_loader
data_loader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

In [ ]:
ims = []
image_key = "observations.images.front_img_1"
actions_key = "actions_cartesian"
for i, data in enumerate(data_loader):
    im = data[image_key][0].permute(1, 2, 0).cpu().numpy() * 255
    im = im.astype(np.uint8)
    actions = data[actions_key][0].cpu().numpy()
    xyz, rot = HPT._extract_xyz(torch.from_numpy(actions))
    im = draw_actions(im, "xyz", "Purples", xyz.numpy(), None, camera_transforms.intrinsics, arm="both")
    ims.append(im)
    if i > 500:
        break

mpy.show_video(ims, fps=30)